# Gold Macro Indicators

Thin notebook entrypoint for macro-domain Gold indicators.

Supported branches:

- `source_system=ecb`: maps Silver ECB FX rates into `gld_macro.dp_macro_indicators`
- `source_system=fred`: collapses Silver FRED revisions to the latest available revision per `(series_id, observation_date)` and maps them into the same Gold table


In [ ]:
import sys
import uuid
from pathlib import Path


def add_repo_src_to_path() -> str:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        src_dir = candidate / "src"
        if (src_dir / "lakehouse" / "__init__.py").exists():
            src_dir_str = str(src_dir)
            if src_dir_str not in sys.path:
                sys.path.insert(0, src_dir_str)
            return src_dir_str

    raise RuntimeError("Could not locate the repository src directory from the current notebook path")


REPO_SRC = add_repo_src_to_path()

from lakehouse.pipelines.gold import run_gold_macro_indicators

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("source_system", "ecb")
dbutils.widgets.text("quote_currencies", "USD")
dbutils.widgets.text("series_ids", "CPIAUCSL,FEDFUNDS,GDP")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
result = run_gold_macro_indicators(
    spark=spark,
    source_system=dbutils.widgets.get("source_system").strip().lower(),
    mode=dbutils.widgets.get("mode").strip().lower(),
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
    run_id=dbutils.widgets.get("run_id").strip(),
    catalog=dbutils.widgets.get("catalog").strip() or "market_macro",
    raw_quote_currencies=dbutils.widgets.get("quote_currencies").strip(),
    raw_series_ids=dbutils.widgets.get("series_ids").strip(),
    display_fn=display,
)

result.as_dict()
